# Phase 1 Load Validation
This notebook validates the loaded data against the phase 0 sample.

In [1]:
import psycopg2
import pandas as pd
import os
import re
from dotenv import load_dotenv

load_dotenv('../.env')

conn = psycopg2.connect(os.environ["DATABASE_URL"])


## Row counts

In [2]:
counts = pd.read_sql("""
SELECT 'raw_posts' as table, count(*) from raw_posts
UNION ALL
SELECT 'influencers', count(*) from influencers
UNION ALL
SELECT 'post_hashtags', count(*) from post_hashtags
""", conn)
display(counts)


C:\Users\aksha\AppData\Local\Temp\ipykernel_24620\2854767936.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  counts = pd.read_sql("""


,table,count
0,raw_posts,1484362
1,influencers,33934
2,post_hashtags,11224531


## Validation against `parsed_sample.csv`

In [3]:
sample_df = pd.read_csv('../data/raw/parsed_sample.csv', dtype={'post_id': str})

# We take the first 100 rows for validation
test_ids = tuple(sample_df['post_id'].head(100).tolist())

query = """
SELECT 
    rp.post_id, 
    rp.username, 
    rp.likes, 
    rp.comments, 
    rp.caption
FROM raw_posts rp
WHERE rp.post_id IN %s
"""

db_df = pd.read_sql(query, conn, params=(test_ids,))

# Calculate hashtag_count from caption to match raw parsing (which counts duplicates)
db_df['hashtag_count'] = db_df['caption'].fillna('').apply(lambda x: len(re.findall(r'#\w+', x)))
db_df['caption_length'] = db_df['caption'].fillna('').apply(len)

# Sort both for comparison and align columns
cols = ['post_id', 'username', 'likes', 'comments', 'caption_length', 'hashtag_count']
csv_subset = sample_df[sample_df['post_id'].isin(test_ids)][cols].sort_values('post_id').reset_index(drop=True)
db_subset = db_df[cols].sort_values('post_id').reset_index(drop=True)

# Replace NaNs with suitable values or handle nulls if necessary
csv_subset['likes'] = csv_subset['likes'].fillna(-1).astype(int)
db_subset['likes'] = db_subset['likes'].fillna(-1).astype(int)
csv_subset['comments'] = csv_subset['comments'].fillna(-1).astype(int)
db_subset['comments'] = db_subset['comments'].fillna(-1).astype(int)
csv_subset['caption_length'] = csv_subset['caption_length'].fillna(0).astype(int)
db_subset['caption_length'] = db_subset['caption_length'].fillna(0).astype(int)
csv_subset['hashtag_count'] = csv_subset['hashtag_count'].fillna(0).astype(int)
db_subset['hashtag_count'] = db_subset['hashtag_count'].fillna(0).astype(int)

# Compare
comparison = csv_subset == db_subset
print(f"Are all selected columns equal? {comparison.all().all()}")
if not comparison.all().all():
    print("Mismatches found!")
    display(csv_subset.compare(db_subset))
else:
    print("Database matches Phase 0 parsed_sample.csv exactly for the test rows.")


Are all selected columns equal? True
Database matches Phase 0 parsed_sample.csv exactly for the test rows.


C:\Users\aksha\AppData\Local\Temp\ipykernel_24620\4153864820.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  db_df = pd.read_sql(query, conn, params=(test_ids,))
